In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, DoubleType

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")

def S(*fields):
    """Build a struct from (name, type) pairs."""
    return StructType([StructField(n, t) for n, t in fields])

reading = S(
    ("station", S(("id", StringType()), ("name", StringType()), ("townCenter", StringType()))),
    ("location", S(("latitude", StringType()), ("longitude", StringType()))),
    ("wbgt", StringType()),
    ("heatStress", StringType()),
)
record = S(
    ("datetime", StringType()),
    ("item", S(("type", StringType()), ("readings", ArrayType(reading)))),
)
WBGT_SCHEMA = S(("data", S(("records", ArrayType(record)))))

wbgt = (spark.table("workspace.heatcheck.bronze_wbgt")
    .select(F.from_json("raw_json", WBGT_SCHEMA).alias("j"))
    .select(F.explode("j.data.records").alias("rec"))
    .select(
        F.to_timestamp("rec.datetime").alias("reading_time"),
        F.col("rec.item.type").alias("reading_type"),
        F.explode("rec.item.readings").alias("r"))
    .select(
        "reading_time", "reading_type",
        F.col("r.station.id").alias("station_id"),
        F.col("r.station.name").alias("station_name"),
        F.col("r.station.townCenter").alias("town_centre"),
        F.col("r.location.latitude").cast("double").alias("latitude"),
        F.col("r.location.longitude").cast("double").alias("longitude"),
        F.col("r.wbgt").cast("double").alias("wbgt"),
        F.col("r.heatStress").alias("heat_stress"))
    .dropDuplicates(["station_id", "reading_time"]))

total = wbgt.count()
clean = wbgt.filter(F.col("wbgt").between(15, 40))
print(f"WBGT rows: {total}, dropped as unrealistic: {total - clean.count()}")

(clean.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.heatcheck.silver_wbgt"))

In [0]:
station = S(("id", StringType()), ("name", StringType()),
            ("location", S(("latitude", DoubleType()), ("longitude", DoubleType()))))
air_reading = S(("timestamp", StringType()),
                ("data", ArrayType(S(("stationId", StringType()), ("value", DoubleType())))))
AIR_SCHEMA = S(("data", S(("stations", ArrayType(station)), ("readings", ArrayType(air_reading)))))

air_raw = (spark.table("workspace.heatcheck.bronze_air_temperature")
    .select(F.from_json("raw_json", AIR_SCHEMA).alias("j")))

stations = (air_raw.select(F.explode("j.data.stations").alias("s"))
    .select(
        F.col("s.id").alias("station_id"),
        F.col("s.name").alias("station_name"),
        F.col("s.location.latitude").alias("latitude"),
        F.col("s.location.longitude").alias("longitude"))
    .dropDuplicates(["station_id"]))

readings = (air_raw.select(F.explode("j.data.readings").alias("rd"))
    .select(F.to_timestamp("rd.timestamp").alias("reading_time"),
            F.explode("rd.data").alias("d"))
    .select("reading_time",
            F.col("d.stationId").alias("station_id"),
            F.col("d.value").alias("air_temp_c")))

air = (readings.join(stations, "station_id", "left")
    .dropDuplicates(["station_id", "reading_time"])
    .filter(F.col("air_temp_c").between(15, 45)))

print("Air temperature rows:", air.count())
(air.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.heatcheck.silver_air_temperature"))

In [0]:
%sql
SELECT reading_time, station_name, wbgt, heat_stress
FROM workspace.heatcheck.silver_wbgt
ORDER BY reading_time DESC, wbgt DESC
LIMIT 20